# TicketRouter: Automated Kaggle GPU (T4) Fine-Tuning Run

This self-contained notebook runs full LoRA SFT on **Qwen/Qwen2.5-1.5B-Instruct** for **Banking77** (77 intent classes) using TRL SFTTrainer and fp16 precision.

In [ ]:
!nvidia-smi

In [ ]:
# Uninstall conflicting torchao version if present and install required packages
!pip uninstall -y torchao
!pip install -q datasets transformers peft trl accelerate mlflow scikit-learn

In [ ]:
import sys
from pathlib import Path

# Guard against torchao in current process
sys.modules["torchao"] = None

Path("ticketrouter").mkdir(exist_ok=True)
with open("ticketrouter/__init__.py", "w") as f:
    f.write('__version__ = "0.1.0"\n')

with open("ticketrouter/data.py", "w") as f:
    f.write(
        '"""Data processing, audit, split creation, and drift synthesis for Banking77."""\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport random\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom datasets import load_dataset\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger(__name__)\n\nSLANG_REPLACEMENTS = {\n    r"\\bplease\\b": "pls",\n    r"\\baccount\\b": "accnt",\n    r"\\bcannot\\b": "cant",\n    r"\\bcan\'t\\b": "cant",\n    r"\\bdo not\\b": "dont",\n    r"\\bdon\'t\\b": "dont",\n    r"\\btransfer\\b": "xfer",\n    r"\\btransferred\\b": "xfered",\n    r"\\bworking\\b": "wrkng",\n    r"\\bthanks\\b": "thx",\n    r"\\bthank you\\b": "thx",\n    r"\\btomorrow\\b": "tmrw",\n    r"\\bapplication\\b": "app",\n    r"\\bdeclined\\b": "declnd",\n    r"\\bavailable\\b": "avail",\n    r"\\bwithdraw\\b": "w/draw",\n    r"\\bwithdrawal\\b": "w/drawal",\n    r"\\bwithdrawing\\b": "w/drawing",\n    r"\\btransaction\\b": "tx",\n    r"\\btransactions\\b": "txs",\n    r"\\bidentity\\b": "id",\n    r"\\bidentification\\b": "id",\n    r"\\bverification\\b": "verif",\n    r"\\bcard\\b": "crd",\n    r"\\bpayment\\b": "pmt",\n    r"\\bpayments\\b": "pmts",\n    r"\\bbalance\\b": "bal",\n    r"\\bdeposit\\b": "dpst",\n}\n\n\ndef normalize_text(text: str) -> str:\n    """Normalize text for near-duplicate detection."""\n    text = text.lower().strip()\n    text = re.sub(r"[^\\w\\s]", "", text)\n    text = re.sub(r"\\s+", " ", text)\n    return text.strip()\n\n\ndef perturb_text(text: str, rng: random.Random) -> str:\n    """Apply realistic typos, lowercase slang, missing apostrophes, and abbreviations."""\n    perturbed = text.lower()\n\n    # Apply slang replacements probabilistically\n    for pattern, replacement in SLANG_REPLACEMENTS.items():\n        if re.search(pattern, perturbed, flags=re.IGNORECASE) and rng.random() < 0.75:\n            perturbed = re.sub(pattern, replacement, perturbed, flags=re.IGNORECASE)\n\n    # Remove apostrophes\n    perturbed = perturbed.replace("\'", "").replace("\u2019", "")\n\n    # Inject minor character typos (adjacent swap, deletion, or double letter)\n    words = perturbed.split()\n    new_words = []\n    for word in words:\n        if len(word) > 4 and rng.random() < 0.35:\n            typo_type = rng.choice(["swap", "drop", "double"])\n            idx = rng.randint(1, len(word) - 2)\n            if typo_type == "swap":\n                word = word[:idx] + word[idx + 1] + word[idx] + word[idx + 2 :]\n            elif typo_type == "drop":\n                word = word[:idx] + word[idx + 1 :]\n            elif typo_type == "double":\n                word = word[:idx] + word[idx] + word[idx:]\n        new_words.append(word)\n\n    return " ".join(new_words)\n\n\ndef audit_dataset(\n    train_df: pd.DataFrame, test_df: pd.DataFrame, label_names: list[str]\n) -> dict[str, Any]:\n    """Audit class balance, data leakage across splits, and likely label noise."""\n    logger.info("Auditing Banking77 dataset...")\n\n    # 1. Class Balance Audit\n    train_counts = train_df["label"].value_counts().sort_index().to_dict()\n    test_counts = test_df["label"].value_counts().sort_index().to_dict()\n\n    class_balance = {\n        "num_classes": len(label_names),\n        "train_min_samples": int(min(train_counts.values())),\n        "train_max_samples": int(max(train_counts.values())),\n        "train_median_samples": float(np.median(list(train_counts.values()))),\n        "test_min_samples": int(min(test_counts.values())),\n        "test_max_samples": int(max(test_counts.values())),\n    }\n\n    # 2. Leakage Audit (Exact and Near Duplicates)\n    train_exact = set(train_df["text"])\n    exact_leakage = [row for row in test_df.to_dict(orient="records") if row["text"] in train_exact]\n\n    train_norm_map: dict[str, int] = {\n        normalize_text(t): int(lbl) for t, lbl in zip(train_df["text"], train_df["label"])\n    }\n    near_leakage = []\n    for row in test_df.to_dict(orient="records"):\n        norm = normalize_text(row["text"])\n        if norm in train_norm_map:\n            near_leakage.append(\n                {\n                    "test_text": row["text"],\n                    "test_label": label_names[row["label"]],\n                    "train_label": label_names[train_norm_map[norm]],\n                    "label_matches": bool(row["label"] == train_norm_map[norm]),\n                }\n            )\n\n    logger.info(\n        f"Found {len(exact_leakage)} exact leakages and {len(near_leakage)} normalized near-leakages."\n    )\n\n    # 3. Label Noise Detection via Stratified Cross-Validation\n    logger.info("Running cross-validation to detect likely label noise in training set...")\n    texts = train_df["text"].tolist()\n    labels = np.array(train_df["label"].tolist())\n\n    min_class_samples = int(min(train_counts.values())) if train_counts else 0\n    n_splits = max(2, min(5, min_class_samples)) if min_class_samples >= 2 else 0\n\n    label_noise_candidates = []\n    if n_splits >= 2:\n        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)\n        oof_preds = np.zeros(len(labels), dtype=int)\n        oof_probs = np.zeros(len(labels), dtype=float)\n\n        for train_idx, val_idx in skf.split(texts, labels):\n            vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)\n            X_tr = vec.fit_transform([texts[i] for i in train_idx])\n            y_tr = labels[train_idx]\n\n            clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)\n            clf.fit(X_tr, y_tr)\n\n            X_val = vec.transform([texts[i] for i in val_idx])\n            probs = clf.predict_proba(X_val)\n            preds = np.argmax(probs, axis=1)\n\n            oof_preds[val_idx] = preds\n            oof_probs[val_idx] = np.max(probs, axis=1)\n\n        for idx, (true_lbl, pred_lbl, prob) in enumerate(zip(labels, oof_preds, oof_probs)):\n            if true_lbl != pred_lbl and prob >= 0.60:\n                label_noise_candidates.append(\n                    {\n                        "index": int(idx),\n                        "text": texts[idx],\n                        "true_label": label_names[true_lbl],\n                        "predicted_label": label_names[pred_lbl],\n                        "confidence": float(prob),\n                    }\n                )\n\n        label_noise_candidates.sort(key=lambda x: x["confidence"], reverse=True)\n        logger.info(\n            f"Identified {len(label_noise_candidates)} likely label-noise training samples (CV conf >= 0.60)."\n        )\n\n    report = {\n        "class_balance": class_balance,\n        "leakage": {\n            "exact_leakage_count": len(exact_leakage),\n            "near_leakage_count": len(near_leakage),\n            "sample_near_leakages": near_leakage[:5],\n        },\n        "label_noise": {\n            "num_candidates": len(label_noise_candidates),\n            "top_candidates": label_noise_candidates[:10],\n        },\n    }\n    return report\n\n\ndef load_and_prepare_data(\n    val_ratio: float = 0.15,\n    seed: int = 42,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], dict[str, Any]]:\n    """Load Banking77, audit, remove leakage, create stratified train/val/test and drifted test set."""\n    ds = load_dataset("PolyAI/banking77")\n    label_names: list[str] = ds["train"].features["label"].names\n\n    train_df = pd.DataFrame(ds["train"])\n    test_df = pd.DataFrame(ds["test"])\n\n    # Run full audit\n    audit_results = audit_dataset(train_df, test_df, label_names)\n\n    # 1. Remove Leakage from Test Set\n    train_norm_set = {normalize_text(t) for t in train_df["text"]}\n    clean_test_mask = [normalize_text(t) not in train_norm_set for t in test_df["text"]]\n    clean_test_df = test_df[clean_test_mask].copy().reset_index(drop=True)\n    clean_test_df["label_name"] = [label_names[lbl] for lbl in clean_test_df["label"]]\n\n    logger.info(\n        f"Removed {len(test_df) - len(clean_test_df)} leaking queries from test set. Clean test size: {len(clean_test_df)}"\n    )\n\n    # 2. Stratified Train / Val split from training data\n    train_split_df, val_split_df = train_test_split(\n        train_df,\n        test_size=val_ratio,\n        stratify=train_df["label"],\n        random_state=seed,\n    )\n    train_split_df = train_split_df.copy().reset_index(drop=True)\n    val_split_df = val_split_df.copy().reset_index(drop=True)\n\n    train_split_df["label_name"] = [label_names[lbl] for lbl in train_split_df["label"]]\n    val_split_df["label_name"] = [label_names[lbl] for lbl in val_split_df["label"]]\n\n    # 3. Create Drift Scenario: "shifted" test set\n    # 15 held-out intents become rare (85% reduction)\n    # Remaining 62 intents represent the primary traffic\n    rng = random.Random(seed)\n    all_label_indices = list(range(len(label_names)))\n    rng.shuffle(all_label_indices)\n    held_out_15_intents = set(all_label_indices[:15])\n\n    shifted_rows = []\n    for _, row in clean_test_df.iterrows():\n        lbl = int(row["label"])\n        # If in held-out 15 intents, downsample heavily\n        if lbl in held_out_15_intents and rng.random() > 0.15:\n            continue\n\n        # Perturb query text with typos, slang, missing apostrophes\n        p_text = perturb_text(row["text"], rng)\n        shifted_rows.append(\n            {\n                "text": p_text,\n                "original_text": row["text"],\n                "label": lbl,\n                "label_name": label_names[lbl],\n                "is_held_out_intent": bool(lbl in held_out_15_intents),\n            }\n        )\n\n    # Oversample the boosted intents to maintain a realistic traffic volume (~3000 requests)\n    boosted_rows = [r for r in shifted_rows if not r["is_held_out_intent"]]\n    target_count = len(clean_test_df)\n    while len(shifted_rows) < target_count:\n        sampled_row = dict(rng.choice(boosted_rows))\n        sampled_row["text"] = perturb_text(sampled_row["original_text"], rng)\n        shifted_rows.append(sampled_row)\n\n    shifted_test_df = pd.DataFrame(shifted_rows).reset_index(drop=True)\n    logger.info(\n        f"Created shifted test set with {len(shifted_test_df)} samples, "\n        f"15 down-sampled intents: {[label_names[i] for i in sorted(held_out_15_intents)]}"\n    )\n\n    audit_results["drift_config"] = {\n        "held_out_15_intents": [label_names[i] for i in sorted(held_out_15_intents)],\n        "shifted_test_samples": len(shifted_test_df),\n        "clean_test_samples": len(clean_test_df),\n        "train_samples": len(train_split_df),\n        "val_samples": len(val_split_df),\n    }\n\n    return train_split_df, val_split_df, clean_test_df, shifted_test_df, label_names, audit_results\n\n\ndef save_splits(\n    output_dir: Path,\n    train_df: pd.DataFrame,\n    val_df: pd.DataFrame,\n    test_clean_df: pd.DataFrame,\n    test_shifted_df: pd.DataFrame,\n    label_names: list[str],\n    audit_report: dict[str, Any],\n) -> None:\n    """Save processed datasets and metadata to JSONL/JSON."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    train_df.to_json(output_dir / "train.jsonl", orient="records", lines=True)\n    val_df.to_json(output_dir / "val.jsonl", orient="records", lines=True)\n    test_clean_df.to_json(output_dir / "test_clean.jsonl", orient="records", lines=True)\n    test_shifted_df.to_json(output_dir / "test_shifted.jsonl", orient="records", lines=True)\n\n    with open(output_dir / "label_names.json", "w", encoding="utf-8") as f:\n        json.dump(label_names, f, indent=2)\n\n    with open(output_dir / "audit_report.json", "w", encoding="utf-8") as f:\n        json.dump(audit_report, f, indent=2)\n\n    logger.info(f"All datasets and audit reports saved successfully to {output_dir}")\n\n\ndef main() -> None:\n    """CLI entrypoint for data preparation."""\n    import argparse\n\n    parser = argparse.ArgumentParser(description="Prepare Banking77 dataset splits and audit.")\n    parser.add_argument("--output-dir", type=str, default="data", help="Output directory")\n    parser.add_argument("--val-ratio", type=float, default=0.15, help="Validation ratio")\n    parser.add_argument("--seed", type=int, default=42, help="Random seed")\n    args = parser.parse_args()\n\n    train_df, val_df, test_clean, test_shifted, label_names, audit = load_and_prepare_data(\n        val_ratio=args.val_ratio, seed=args.seed\n    )\n    save_splits(\n        Path(args.output_dir), train_df, val_df, test_clean, test_shifted, label_names, audit\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
    )

with open("ticketrouter/train.py", "w") as f:
    f.write(
        '"""LoRA Supervised Fine-Tuning (SFT) for intent classification using TRL."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport difflib\nimport json\nimport logging\nimport os\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\n# Prevent incompatible torchao (< 0.16.0) preinstalled in Kaggle from breaking PEFT dispatchers\ntry:\n    import torchao\n    from packaging import version\n\n    if version.parse(getattr(torchao, "__version__", "0.0.0")) < version.parse("0.16.0"):\n        sys.modules["torchao"] = None\nexcept Exception:\n    pass\n\nimport mlflow\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom datasets import Dataset\nfrom peft import LoraConfig, TaskType, get_peft_model\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    TrainerCallback,\n)\nfrom trl import SFTConfig, SFTTrainer\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger(__name__)\n\nSYSTEM_PROMPT = (\n    "You are an enterprise banking query intent classifier. "\n    "Classify the customer inquiry into exactly one of the 77 standard intents. "\n    "Output ONLY the exact intent label."\n)\n\n\ndef format_instruction(text: str, label_name: str | None = None) -> str:\n    """Format prompt with ChatML style for Qwen models."""\n    prompt = (\n        f"<|im_start|>system\\n{SYSTEM_PROMPT}<|im_end|>\\n"\n        f"<|im_start|>user\\nQuery: {text}<|im_end|>\\n"\n        f"<|im_start|>assistant\\n"\n    )\n    if label_name is not None:\n        return prompt + f"{label_name}<|im_end|>"\n    return prompt\n\n\ndef map_to_nearest_label(\n    raw_output: str, valid_labels: list[str], valid_labels_set: set[str]\n) -> tuple[str, bool]:\n    """Map raw model output to nearest valid intent. Returns (mapped_label, is_invalid_raw)."""\n    # Clean output: first line, lowercased, stripped punctuation\n    cleaned = raw_output.strip().split("\\n")[0].strip().lower()\n    cleaned = cleaned.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()\n    if cleaned.startswith("intent:"):\n        cleaned = cleaned[7:].strip()\n\n    if cleaned in valid_labels_set:\n        return cleaned, False\n\n    matches = difflib.get_close_matches(cleaned, valid_labels, n=1, cutoff=0.0)\n    nearest = matches[0] if matches else valid_labels[0]\n    return nearest, True\n\n\nclass MLflowLossCallback(TrainerCallback):\n    """Custom callback to stream step-level training loss to MLflow."""\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if logs and mlflow.active_run():\n            for k, v in logs.items():\n                if isinstance(v, (int, float)):\n                    mlflow.log_metric(k, v, step=state.global_step)\n\n\ndef evaluate_model(\n    model: torch.nn.Module,\n    tokenizer: AutoTokenizer,\n    eval_df: pd.DataFrame,\n    valid_labels: list[str],\n    device: torch.device,\n    max_eval_samples: int | None = None,\n) -> dict[str, Any]:\n    """Evaluate accuracy, macro-F1, latency, and invalid-output rate on a dataset split."""\n    if max_eval_samples and len(eval_df) > max_eval_samples:\n        eval_df = eval_df.iloc[:max_eval_samples].copy()\n\n    valid_labels_set = set(valid_labels)\n    model.eval()\n\n    y_true = eval_df["label_name"].tolist()\n    y_pred = []\n    latencies_ms = []\n    invalid_count = 0\n\n    with torch.no_grad():\n        for text in eval_df["text"].tolist():\n            prompt = format_instruction(text)\n            inputs = tokenizer(prompt, return_tensors="pt").to(device)\n\n            t0 = time.perf_counter()\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=20,\n                do_sample=False,\n                pad_token_id=tokenizer.pad_token_id,\n                eos_token_id=tokenizer.eos_token_id,\n            )\n            lat_ms = (time.perf_counter() - t0) * 1000.0\n            latencies_ms.append(lat_ms)\n\n            gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]\n            raw_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)\n\n            mapped_lbl, is_invalid = map_to_nearest_label(raw_text, valid_labels, valid_labels_set)\n            if is_invalid:\n                invalid_count += 1\n            y_pred.append(mapped_lbl)\n\n    acc = float(accuracy_score(y_true, y_pred))\n    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))\n    inv_rate = float(invalid_count / max(1, len(y_true)))\n    p50 = float(np.percentile(latencies_ms, 50))\n    p95 = float(np.percentile(latencies_ms, 95))\n\n    return {\n        "accuracy": acc,\n        "macro_f1": macro_f1,\n        "invalid_output_rate": inv_rate,\n        "latency_p50_ms": p50,\n        "latency_p95_ms": p95,\n        "samples_evaluated": len(y_true),\n    }\n\n\ndef train_lora(\n    base_model_name: str | None = None,\n    data_dir: Path = Path("data"),\n    output_dir: Path = Path("models/qwen_lora_adapter"),\n    epochs: int = 3,\n    batch_size: int = 4,\n    learning_rate: float = 2e-4,\n    smoke: bool = False,\n    tracking_uri: str | None = None,\n    experiment_name: str = "ticketrouter_sft",\n    registered_model_name: str = "ticketrouter_model",\n) -> dict[str, Any]:\n    """Train LoRA fine-tuned model with TRL SFTTrainer and log everything to MLflow."""\n    if base_model_name is None:\n        base_model_name = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")\n    if tracking_uri is None:\n        tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")\n\n    mlflow.set_tracking_uri(tracking_uri)\n    mlflow.set_experiment(experiment_name)\n\n    # Determine Device & Precision\n    is_cuda = torch.cuda.is_available()\n    is_mps = torch.backends.mps.is_available()\n    if is_cuda:\n        device = torch.device("cuda")\n        torch_dtype = torch.float16  # T4-friendly fp16\n        use_fp16 = True\n    elif is_mps and not smoke:\n        device = torch.device("mps")\n        torch_dtype = torch.float32\n        use_fp16 = False\n    else:\n        device = torch.device("cpu")\n        torch_dtype = torch.float32\n        use_fp16 = False\n\n    logger.info(\n        f"Training configuration: device={device}, dtype={torch_dtype}, "\n        f"use_fp16={use_fp16}, smoke={smoke}, base_model={base_model_name}"\n    )\n\n    # Load splits\n    train_df = pd.read_json(data_dir / "train.jsonl", lines=True)\n    val_df = pd.read_json(data_dir / "val.jsonl", lines=True)\n    test_clean_df = pd.read_json(data_dir / "test_clean.jsonl", lines=True)\n    test_shifted_df = pd.read_json(data_dir / "test_shifted.jsonl", lines=True)\n\n    with open(data_dir / "label_names.json", encoding="utf-8") as f:\n        valid_labels = json.load(f)\n\n    if smoke:\n        logger.info("SMOKE MODE: subsampling datasets (200 train, 40 val, 40 test)")\n        train_df = train_df.iloc[:200].copy()\n        val_df = val_df.iloc[:40].copy()\n        test_clean_df = test_clean_df.iloc[:40].copy()\n        test_shifted_df = test_shifted_df.iloc[:40].copy()\n\n    # Create formatted text column\n    train_df["formatted_text"] = [\n        format_instruction(t, lbl) for t, lbl in zip(train_df["text"], train_df["label_name"])\n    ]\n    val_df["formatted_text"] = [\n        format_instruction(t, lbl) for t, lbl in zip(val_df["text"], val_df["label_name"])\n    ]\n\n    train_ds = Dataset.from_pandas(train_df[["formatted_text"]])\n    val_ds = Dataset.from_pandas(val_df[["formatted_text"]])\n\n    # Load Tokenizer\n    logger.info(f"Loading tokenizer: {base_model_name}...")\n    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n\n    # Load Base Model\n    logger.info(f"Loading base model: {base_model_name}...")\n    model = AutoModelForCausalLM.from_pretrained(\n        base_model_name,\n        torch_dtype=torch_dtype,\n        device_map=None if (smoke or not is_cuda) else "auto",\n        trust_remote_code=True,\n    )\n    if smoke or not is_cuda:\n        model.to(device)\n\n    # Configure LoRA: r=16, alpha=32, target all linear projections\n    lora_config = LoraConfig(\n        r=16,\n        lora_alpha=32,\n        lora_dropout=0.05,\n        bias="none",\n        task_type=TaskType.CAUSAL_LM,\n        target_modules=[\n            "q_proj",\n            "k_proj",\n            "v_proj",\n            "o_proj",\n            "gate_proj",\n            "up_proj",\n            "down_proj",\n        ],\n    )\n    model = get_peft_model(model, lora_config)\n    model.print_trainable_parameters()\n\n    max_steps = 20 if smoke else -1\n    effective_epochs = 1 if smoke else epochs\n    grad_accum = 1 if smoke else 2\n    grad_checkpointing = not smoke and is_cuda\n\n    training_args = SFTConfig(\n        output_dir=str(output_dir / "checkpoints"),\n        dataset_text_field="formatted_text",\n        max_length=256,\n        num_train_epochs=effective_epochs,\n        max_steps=max_steps,\n        per_device_train_batch_size=batch_size,\n        gradient_accumulation_steps=grad_accum,\n        learning_rate=learning_rate,\n        fp16=use_fp16,\n        logging_steps=5 if smoke else 20,\n        eval_strategy="no" if smoke else "epoch",\n        save_strategy="no" if smoke else "epoch",\n        gradient_checkpointing=grad_checkpointing,\n        report_to=[],\n    )\n\n    trainer = SFTTrainer(\n        model=model,\n        args=training_args,\n        train_dataset=train_ds,\n        eval_dataset=val_ds if not smoke else None,\n        callbacks=[MLflowLossCallback()],\n    )\n\n    with mlflow.start_run(run_name="qwen_lora_sft") as run:\n        mlflow.log_params(\n            {\n                "base_model": base_model_name,\n                "lora_r": 16,\n                "lora_alpha": 32,\n                "lora_dropout": 0.05,\n                "learning_rate": learning_rate,\n                "epochs": effective_epochs,\n                "batch_size": batch_size,\n                "smoke": smoke,\n                "device": str(device),\n                "fp16": use_fp16,\n            }\n        )\n\n        logger.info("Starting SFT training...")\n        train_result = trainer.train()\n        mlflow.log_metrics(\n            {\n                "train_loss": train_result.training_loss,\n                "train_runtime_sec": train_result.metrics.get("train_runtime", 0.0),\n            }\n        )\n\n        # Save LoRA Adapter\n        output_dir.mkdir(parents=True, exist_ok=True)\n        model.save_pretrained(str(output_dir))\n        tokenizer.save_pretrained(str(output_dir))\n        mlflow.log_artifact(str(output_dir), artifact_path="lora_adapter")\n        logger.info(f"LoRA adapter saved to {output_dir}")\n\n        # Post-training evaluation\n        logger.info("Evaluating fine-tuned model on clean test set...")\n        clean_eval = evaluate_model(\n            model,\n            tokenizer,\n            test_clean_df,\n            valid_labels,\n            device,\n            max_eval_samples=40 if smoke else 500,\n        )\n\n        logger.info("Evaluating fine-tuned model on shifted test set...")\n        shifted_eval = evaluate_model(\n            model,\n            tokenizer,\n            test_shifted_df,\n            valid_labels,\n            device,\n            max_eval_samples=40 if smoke else 500,\n        )\n\n        metrics = {\n            "clean_test_accuracy": clean_eval["accuracy"],\n            "clean_test_macro_f1": clean_eval["macro_f1"],\n            "clean_test_invalid_rate": clean_eval["invalid_output_rate"],\n            "shifted_test_accuracy": shifted_eval["accuracy"],\n            "shifted_test_macro_f1": shifted_eval["macro_f1"],\n            "latency_p50_ms": clean_eval["latency_p50_ms"],\n            "latency_p95_ms": clean_eval["latency_p95_ms"],\n        }\n        mlflow.log_metrics(metrics)\n\n        # Save metrics JSON artifact for gate.py\n        metrics_file = Path("metrics/candidate_metrics.json")\n        metrics_file.parent.mkdir(parents=True, exist_ok=True)\n        with open(metrics_file, "w", encoding="utf-8") as f:\n            json.dump(\n                {\n                    "run_id": run.info.run_id,\n                    "model_name": registered_model_name,\n                    "metrics": metrics,\n                    "clean_eval": clean_eval,\n                    "shifted_eval": shifted_eval,\n                },\n                f,\n                indent=2,\n            )\n        mlflow.log_artifact(str(metrics_file), artifact_path="metrics")\n\n        logger.info(\n            f"Fine-Tuning Run {run.info.run_id} Completed:\\n"\n            f"  Clean Test Acc:   {clean_eval[\'accuracy\']:.4f} (F1: {clean_eval[\'macro_f1\']:.4f})\\n"\n            f"  Shifted Test Acc: {shifted_eval[\'accuracy\']:.4f} (F1: {shifted_eval[\'macro_f1\']:.4f})\\n"\n            f"  Invalid Output:   {clean_eval[\'invalid_output_rate\']:.4f}\\n"\n            f"  Latency p95:      {clean_eval[\'latency_p95_ms\']:.2f} ms"\n        )\n\n        return {\n            "run_id": run.info.run_id,\n            "metrics": metrics,\n            "adapter_dir": str(output_dir),\n        }\n\n\ndef main() -> None:\n    """CLI entrypoint for LoRA SFT."""\n    parser = argparse.ArgumentParser(description="LoRA Fine-Tuning for Banking77 Intent Routing")\n    parser.add_argument("--base-model", type=str, default=None, help="Base model name or path")\n    parser.add_argument(\n        "--data-dir", type=str, default="data", help="Directory with dataset splits"\n    )\n    parser.add_argument(\n        "--output-dir", type=str, default="models/qwen_lora_adapter", help="Output adapter dir"\n    )\n    parser.add_argument("--epochs", type=int, default=3, help="Training epochs")\n    parser.add_argument("--batch-size", type=int, default=4, help="Per device batch size")\n    parser.add_argument("--lr", type=float, default=2e-4, help="Learning rate")\n    parser.add_argument(\n        "--smoke", action="store_true", help="Run 20-step smoke test on 200 samples"\n    )\n    parser.add_argument("--tracking-uri", type=str, default=None, help="MLflow tracking URI")\n    args = parser.parse_args()\n\n    train_lora(\n        base_model_name=args.base_model,\n        data_dir=Path(args.data_dir),\n        output_dir=Path(args.output_dir),\n        epochs=args.epochs,\n        batch_size=args.batch_size,\n        learning_rate=args.lr,\n        smoke=args.smoke,\n        tracking_uri=args.tracking_uri,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
    )
print("Module files written successfully.")

In [ ]:
!python -m ticketrouter.data --output-dir data

In [ ]:
!python -m ticketrouter.train \
    --base-model "Qwen/Qwen2.5-1.5B-Instruct" \
    --data-dir data \
    --output-dir models/qwen_lora_adapter \
    --epochs 3 \
    --batch-size 8 \
    --lr 2e-4

In [ ]:
!tar -czvf /kaggle/working/ticketrouter_lora_adapter.tar.gz models/qwen_lora_adapter metrics/candidate_metrics.json
!ls -lh /kaggle/working/ticketrouter_lora_adapter.tar.gz